# Within t+2 모델링

기존 `5_data_preprocessing.ipynb`에서 생성한 LSTM tensor를 사용해 비교용 ML/MLP baseline과 single-output LSTM을 학습합니다.

- LR/RF/XGB/LightGBM/MLP baseline은 anchor `t` 시점 feature만 입력으로 사용합니다.
- LSTM은 기존과 동일하게 `t-3~t` 4개 time step을 입력으로 사용합니다.
- target은 `t`, `t+1`, `t+2` 중 delirium이 한 번이라도 발생하는지 여부입니다.
- binary within target에는 horizon별 mask가 없으므로 기본값은 full `t~t+2` target이 모두 관측된 row만 사용합니다.


In [ ]:
from pathlib import Path
import importlib.util
import json
import random

import numpy as np
import pandas as pd
import torch


In [ ]:
# 작업 위치
CWD = Path.cwd().resolve()
if CWD.name == "src" and CWD.parent.name == "Parkinson":
    PROJECT_DIR = CWD.parent
elif CWD.name == "Parkinson":
    PROJECT_DIR = CWD
elif (CWD / "Parkinson").exists():
    PROJECT_DIR = CWD / "Parkinson"
else:
    PROJECT_DIR = CWD.parent

SRC_DIR = PROJECT_DIR / "src"
MODELING_DIR = PROJECT_DIR / "processed" / "data_split"
MODEL_DIR = PROJECT_DIR / "models" / "within_t_plus_2"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "modeling" / "within_t_plus_2"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("MODELING_DIR:", MODELING_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# 설정값 (config)
RANDOM_STATE = 42
N_FOLDS = 5
N_TRIALS_ML = 30
N_TRIALS_MLP = 30
N_TRIALS_LSTM = 30
MAX_EPOCHS = 25
PATIENCE = 5

# False이면 target_available_count >= 3인 full t~t+2 window만 사용
ALLOW_PARTIAL_TARGET_WINDOW = False

# 실행할 모델 목록. xgboost/lightgbm package가 없으면 해당 모델은 자동 skip됩니다.
ML_MODEL_NAMES = ["LR", "RF", "XGB", "LGBM"]
RUN_MLP = True
RUN_LSTM = True


In [ ]:
# CUDA GPU 사용 설정
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("device:", device)
    print("gpu:", torch.cuda.get_device_name(0))
    print("cuda version:", torch.version.cuda)
else:
    device = torch.device("cpu")
    print("device:", device)
    print("CUDA GPU를 찾지 못해 CPU로 실행합니다.")

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.set_num_threads(2)


## 모델링 helper 로딩

구현은 같은 폴더의 `6_modeling_within_t_plus_2.py`에 두고, 이 노트북은 기존 notebook 흐름처럼 config, QA, tuning, 저장을 셀 단위로 실행합니다.


In [ ]:
helper_path = SRC_DIR / "6_modeling_within_t_plus_2.py"
spec = importlib.util.spec_from_file_location("within_t_plus_2_helper", helper_path)
within_helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(within_helper)

print("loaded helper:", helper_path)


## 전처리 산출물 로딩


In [ ]:
data = within_helper.load_modeling_data(
    PROJECT_DIR,
    require_full_target_window=not ALLOW_PARTIAL_TARGET_WINDOW,
)

X_train_seq = data.x_train_seq
X_test_seq = data.x_test_seq
X_train_t = data.x_train_t
X_test_t = data.x_test_t
y_train = data.y_train
y_test = data.y_test
meta_train = data.meta_train
meta_test = data.meta_test


In [ ]:
# 로딩된 데이터의 기본 형태와 target 분포 확인
data_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "sequence_samples": X_train_seq.shape[0],
            "sequence_length": X_train_seq.shape[1],
            "n_features": X_train_seq.shape[2],
            "t_point_shape": str(X_train_t.shape),
            "positive_rate": y_train.mean(),
            "nan_count": np.isnan(X_train_seq).sum(),
            "target_window": "partial_allowed" if ALLOW_PARTIAL_TARGET_WINDOW else "full_t_to_t_plus_2_only",
        },
        {
            "split": "test",
            "sequence_samples": X_test_seq.shape[0],
            "sequence_length": X_test_seq.shape[1],
            "n_features": X_test_seq.shape[2],
            "t_point_shape": str(X_test_t.shape),
            "positive_rate": y_test.mean(),
            "nan_count": np.isnan(X_test_seq).sum(),
            "target_window": "partial_allowed" if ALLOW_PARTIAL_TARGET_WINDOW else "full_t_to_t_plus_2_only",
        },
    ]
)

display(data_summary)
display(meta_train["target_available_count"].value_counts().sort_index().rename("train_target_available_count"))
display(meta_test["target_available_count"].value_counts().sort_index().rename("test_target_available_count"))


## Subject-level cross-validation split

같은 환자의 window가 train과 validation fold에 동시에 들어가지 않도록 `subject_id` 기준으로 K-fold CV를 구성합니다.


In [ ]:
cv_folds = within_helper.make_subject_cv_folds(meta_train, y_train, N_FOLDS, RANDOM_STATE)
print(f"Using {len(cv_folds)} subject-level CV folds")

cv_summary = pd.DataFrame(
    [
        {
            "fold_id": fold["fold_id"],
            "train_sequences": int(fold["train_mask"].sum()),
            "validation_sequences": int(fold["val_mask"].sum()),
            "train_subjects": fold["n_train_subjects"],
            "validation_subjects": fold["n_val_subjects"],
            "train_positive_rate": fold["train_positive_rate"],
            "validation_positive_rate": fold["val_positive_rate"],
        }
        for fold in cv_folds
    ]
)

display(cv_summary)


## ML baseline hyperparameter tuning

LR/RF/XGB/LightGBM은 모두 anchor `t` 시점 feature만 사용합니다. XGB와 LightGBM은 CUDA가 사용 가능하면 GPU parameter를 우선 적용합니다.


In [ ]:
availability = pd.DataFrame(
    [
        {"model": "XGB", "available": within_helper.optional_import("xgboost") is not None},
        {"model": "LightGBM", "available": within_helper.optional_import("lightgbm") is not None},
        {"model": "CUDA", "available": torch.cuda.is_available()},
    ]
)
display(availability)


In [ ]:
ml_test_metric_rows = []
for model_name in ML_MODEL_NAMES:
    print()
    print(f"=== {model_name}: current t-point ML baseline ===")
    result = within_helper.run_ml_tuning(
        model_name=model_name,
        data=data,
        folds=cv_folds,
        output_dir=OUTPUT_DIR,
        model_dir=MODEL_DIR,
        n_trials=N_TRIALS_ML,
        random_state=RANDOM_STATE,
        use_gpu=device.type == "cuda",
    )
    if result is not None:
        ml_test_metric_rows.append(result)

ml_test_metrics = pd.DataFrame(ml_test_metric_rows)
if not ml_test_metrics.empty:
    display(ml_test_metrics.sort_values("auprc", ascending=False))


## MLP baseline hyperparameter tuning

MLP는 LR/RF/XGB/LightGBM과 동일하게 anchor `t` 시점 feature만 사용하는 PyTorch deep learning baseline입니다. CUDA가 있으면 GPU/AMP를 사용합니다.


In [ ]:
if RUN_MLP:
    print("=== MLP: current t-point deep learning baseline ===")
    mlp_test_metrics = within_helper.run_mlp_tuning(
        data=data,
        folds=cv_folds,
        output_dir=OUTPUT_DIR,
        model_dir=MODEL_DIR,
        n_trials=N_TRIALS_MLP,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        random_state=RANDOM_STATE,
        device=device,
        amp_enabled=device.type == "cuda",
    )
    display(pd.DataFrame([mlp_test_metrics]))
else:
    mlp_test_metrics = None


## Single-output LSTM hyperparameter tuning

기존 encoder-decoder multi-horizon 구조 대신, `t-3~t` input sequence를 LSTM이 읽고 마지막 hidden state에서 `within_t_plus_2` logit 하나를 출력합니다.


In [ ]:
if RUN_LSTM:
    lstm_test_metrics = within_helper.run_lstm_tuning(
        data=data,
        folds=cv_folds,
        output_dir=OUTPUT_DIR,
        model_dir=MODEL_DIR,
        n_trials=N_TRIALS_LSTM,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        random_state=RANDOM_STATE,
        device=device,
        amp_enabled=device.type == "cuda",
    )
    display(pd.DataFrame([lstm_test_metrics]))
else:
    lstm_test_metrics = None


## 최종 결과 요약 저장


In [ ]:
summary_rows = []
if not ml_test_metrics.empty:
    summary_rows.extend(ml_test_metrics.to_dict(orient="records"))
if mlp_test_metrics is not None:
    summary_rows.append(mlp_test_metrics)
if lstm_test_metrics is not None:
    summary_rows.append(lstm_test_metrics)

within_t_plus_2_test_summary = pd.DataFrame(summary_rows)
if not within_t_plus_2_test_summary.empty:
    within_t_plus_2_test_summary = within_t_plus_2_test_summary.sort_values("auprc", ascending=False).reset_index(drop=True)
    within_t_plus_2_test_summary.to_csv(OUTPUT_DIR / "within_t_plus_2_test_metrics_summary.csv", index=False)
    display(within_t_plus_2_test_summary)

print("Saved within t+2 ML/MLP baselines and single-output LSTM outputs")
